In [14]:
import hashlib
import random
import sys
import time

# 1. 가상 데이터 10만 개 생성
with open("stream_data.txt", "w") as f:
    for _ in range(100000):
        user_id = f"user_{random.randint(1, 20000)}"
        f.write(f"{user_id}\n")

In [15]:
# 2. 알고리즘 구현
class BloomFilter:

    def __init__(self, size, num_hashes):
        self.size = size
        self.num_hashes = num_hashes
        self.bit_array = [0] * size

    def _hashes(self, item):
        for i in range(self.num_hashes):
            hash_res = hashlib.md5(f"{item}_{i}".encode()).hexdigest()
            yield int(hash_res, 16) % self.size

    def add(self, item):
        for h in self._hashes(item):
            self.bit_array[h] = 1

    def contains(self, item):
        for h in self._hashes(item):
            if self.bit_array[h] == 0:
                return False
        return True


class CountMinSketch:

    def __init__(self, width, depth):
        self.width = width
        self.depth = depth
        self.table = [[0] * width for _ in range(depth)]

    def _hashes(self, item):
        for i in range(self.depth):
            hash_res = hashlib.sha1(f"{item}_{i}".encode()).hexdigest()
            yield int(hash_res, 16) % self.width

    def add(self, item):
        for i, h in enumerate(self._hashes(item)):
            self.table[i][h] += 1

    def estimate(self, item):
        return min(self.table[i][h] for i, h in enumerate(self._hashes(item)))

In [16]:
# 3. 실험 함수 정의

def run_experiment(bf_size, bf_hashes, cms_width, cms_depth):
    bf = BloomFilter(size=bf_size, num_hashes=bf_hashes)
    cms = CountMinSketch(width=cms_width, depth=cms_depth)

    gt_set = set()
    gt_dict = {}

    start_time = time.time()

    with open("stream_data.txt", "r") as f:
        for line in f:
            item = line.strip()
            gt_set.add(item)
            gt_dict[item] = gt_dict.get(item, 0) + 1
            bf.add(item)
            cms.add(item)

    end_time = time.time()

    duration = end_time - start_time
    throughput = 100000 / duration

    bf_mem = sys.getsizeof(bf.bit_array)
    cms_mem = sys.getsizeof(cms.table) * cms_depth
    gt_mem = sys.getsizeof(gt_set) + sys.getsizeof(gt_dict)

    test_items = [f"user_{i}" for i in range(1, 1000)]
    fp_count = 0
    cms_error_sum = 0

    for item in test_items:
        actual_exist = item in gt_set
        bf_exist = bf.contains(item)
        if not actual_exist and bf_exist:
            fp_count += 1

        actual_count = gt_dict.get(item, 0)
        cms_count = cms.estimate(item)
        cms_error_sum += abs(actual_count - cms_count)

    fp_rate = (
        fp_count / len(test_items) if (len(test_items) - len(gt_set)) > 0 else 0
    )
    avg_cms_error = cms_error_sum / len(test_items)

    return {
        "time": duration,
        "throughput": throughput,
        "bf_mem": bf_mem,
        "cms_mem": cms_mem,
        "gt_mem": gt_mem,
        "bf_fp_rate": fp_rate,
        "cms_error": avg_cms_error,
    }

In [17]:
# 4. 실험 실행 및 결과 출력
print("--- 실험 1 진행 중 ---")
res1 = run_experiment(bf_size=50000, bf_hashes=3, cms_width=500, cms_depth=3)

print("--- 실험 2 진행 중 ---")
res2 = run_experiment(
    bf_size=200000, bf_hashes=5, cms_width=2000, cms_depth=5
)

print("\n================ 실험 결과 비교 ================")
print(f"{'분석 항목':<15} | {'실험 1 (작은 메모리)':<20} | {'실험 2 (큰 메모리)':<20}")
print("-" * 65)
print(f"{'전체 처리 시간':<15} | {res1['time']:.4f}초 {'':<11} | {res2['time']:.4f}초")
print(
    f"{'초당 처리량':<15} | {res1['throughput']:.1f} ops/s {'':<4} | {res2['throughput']:.1f} ops/s"
)
print(
    f"{'BF 메모리 크기':<15} | {res1['bf_mem']:,} bytes {'':<5} | {res2['bf_mem']:,} bytes"
)
print(
    f"{'CMS 메모리 크기':<15} | {res1['cms_mem']:,} bytes {'':<4} | {res2['cms_mem']:,} bytes"
)
print(
    f"{'GT(정답) 메모리':<15} | {res1['gt_mem']:,} bytes {'':<5} | {res2['gt_mem']:,} bytes"
)
print(
    f"{'BF 오탐률(FP)':<15} | {res1['bf_fp_rate']*100:.2f}% {'':<11} | {res2['bf_fp_rate']*100:.2f}%"
)
print(
    f"{'CMS 평균오차':<15} | {res1['cms_error']:.2f} {'':<13} | {res2['cms_error']:.2f}"
)

--- 실험 1 진행 중 ---
--- 실험 2 진행 중 ---

================ 실험 결과 비교 ================
분석 항목           | 실험 1 (작은 메모리)        | 실험 2 (큰 메모리)        
-----------------------------------------------------------------
전체 처리 시간        | 1.0122초             | 1.6311초
초당 처리량          | 98797.2 ops/s      | 61306.9 ops/s
BF 메모리 크기       | 400,056 bytes       | 1,600,056 bytes
CMS 메모리 크기      | 264 bytes      | 600 bytes
GT(정답) 메모리      | 2,512,520 bytes       | 2,512,520 bytes
BF 오탐률(FP)      | 0.00%             | 0.00%
CMS 평균오차        | 170.75               | 30.50
